In [1]:
# ===== IMPORTS =====
import numpy as np
import pandas as pd
import json
from scipy.spatial import cKDTree
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv

In [2]:
# ===== LOAD DATA =====
processed_df = pd.read_csv("processed_data.csv", low_memory=False)

In [3]:
# ===== SELECT GNN-RELEVANT COLUMNS =====
gnn_columns = [
    "Accident_Index", "Latitude", "Longitude",
    "1st_Road_Number", "2nd_Road_Number", "Speed_limit",
    "Year", "Month", "Day", "Hour", "Is_Weekend", "Is_Rush_Hour",
    "Is_Peak_Hour", "Is_Night", "Is_Morning", "Is_Evening",
    "Bad_Weather_Flag", "Poor_Visibility_Flag",
]
prefixes = [
    "Road_Type_", "1st_Road_Class_", "2nd_Road_Class_",
    "Junction_Control_", "Junction_Detail_", "Weather_Conditions_",
    "Road_Surface_Conditions_", "Light_Conditions_",
    "Urban_or_Rural_Area_", "Traffic_Density_Indicator_", "Season_"
]
for col in processed_df.columns:
    for prefix in prefixes:
        if col.startswith(prefix):
            if col not in gnn_columns:
                gnn_columns.append(col)
            break

gnn_df = processed_df[gnn_columns].copy()

In [4]:
# ===== FILTER TO JUNCTIONS, GRID-SNAP, SCOPE TO 2017 =====
junction_col = "Junction_Detail_Not At Junction Or Within 20 Metres"
at_junction_df = gnn_df[gnn_df[junction_col] == 0].copy()
at_junction_df["node_lat"] = at_junction_df["Latitude"].round(3)
at_junction_df["node_lon"] = at_junction_df["Longitude"].round(3)
at_junction_df["node_id"] = at_junction_df["node_lat"].astype(str) + "_" + at_junction_df["node_lon"].astype(str)
year_filtered_df = at_junction_df[at_junction_df["Year"] == 2017].copy()


In [5]:
# ===== NODE FEATURES =====
node_features = year_filtered_df.groupby("node_id").agg(
    latitude=("node_lat", "first"), longitude=("node_lon", "first"),
    avg_speed_limit=("Speed_limit", "mean"), accident_count=("Accident_Index", "count"),
    bad_weather_rate=("Bad_Weather_Flag", "mean"), poor_visibility_rate=("Poor_Visibility_Flag", "mean"),
).reset_index()
node_features["node_index"] = range(len(node_features))
road_at_node = year_filtered_df.groupby("node_id")["1st_Road_Number"].agg(lambda x: x.mode()[0])
node_features["road_number"] = node_features["node_id"].map(road_at_node)


In [6]:
# ===== EDGES (nearest-neighbor chain per road) =====
def order_nodes_by_nearest_neighbor(group_df):
    remaining = group_df.copy().reset_index(drop=True)
    start_row = remaining.sort_values(["latitude", "longitude"]).iloc[0]
    chain = [start_row["node_index"]]
    remaining = remaining[remaining["node_index"] != start_row["node_index"]]
    current = start_row
    while len(remaining) > 0:
        dists = np.sqrt((remaining["latitude"]-current["latitude"])**2 + (remaining["longitude"]-current["longitude"])**2)
        nearest_pos = dists.idxmin()
        nearest_row = remaining.loc[nearest_pos]
        chain.append(nearest_row["node_index"])
        remaining = remaining.drop(nearest_pos)
        current = nearest_row
    return chain

edges = []
for road_num, group in node_features.groupby("road_number"):
    if road_num == 0 or len(group) < 2:
        continue
    ordered_nodes = order_nodes_by_nearest_neighbor(group)
    for i in range(len(ordered_nodes) - 1):
        edges.append((ordered_nodes[i], ordered_nodes[i + 1]))

node_features_sorted = node_features.sort_values("node_index").reset_index(drop=True)

In [7]:
# ===== EDGE LABELS (nearest-segment accident matching, ALL 2017 accidents) =====
edge_midpoints = []
for node_a, node_b in edges:
    la, lo = node_features_sorted.iloc[node_a][["latitude","longitude"]]
    lb, lob = node_features_sorted.iloc[node_b][["latitude","longitude"]]
    edge_midpoints.append(((la+lb)/2, (lo+lob)/2))
edge_midpoints = np.array(edge_midpoints)
edge_tree = cKDTree(edge_midpoints)

all_2017_df = gnn_df[gnn_df["Year"] == 2017].dropna(subset=["Latitude","Longitude"]).copy()
_, nearest_edge_idx = edge_tree.query(all_2017_df[["Latitude","Longitude"]].values)
all_2017_df["nearest_edge"] = nearest_edge_idx
edge_accident_counts = all_2017_df.groupby("nearest_edge").size()

new_edge_labels = np.zeros(len(edges))
for edge_i, count in edge_accident_counts.items():
    new_edge_labels[edge_i] = count
new_edge_labels = new_edge_labels / new_edge_labels.max()
print("LABEL CHECK (std should be ~0.06):", new_edge_labels.std())

LABEL CHECK (std should be ~0.06): 0.06415529263755282


In [8]:
# ===== GRAPH OBJECT =====
feature_columns = ["latitude","longitude","avg_speed_limit","accident_count","bad_weather_rate","poor_visibility_rate"]
scaler = StandardScaler()
X = torch.tensor(scaler.fit_transform(node_features_sorted[feature_columns]), dtype=torch.float)

edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
y_edge = torch.tensor(new_edge_labels, dtype=torch.float)
y_edge = torch.cat([y_edge, y_edge])
data = Data(x=X, edge_index=edge_index)

In [9]:

# ===== MODEL =====
class RoadRiskGNN(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.edge_predictor = nn.Linear(hidden_dim * 2, 1)
    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        h = F.relu(self.conv2(h, edge_index))
        return h
    def predict_edges(self, node_embeddings, edge_index):
        combined = torch.cat([node_embeddings[edge_index[0]], node_embeddings[edge_index[1]]], dim=1)
        return torch.sigmoid(self.edge_predictor(combined)).squeeze()


In [ ]:
# ===== TRAIN (leak-free split, more capacity/epochs, LR decay) =====
num_original_edges = len(edges)
train_orig, test_orig = train_test_split(list(range(num_original_edges)), test_size=0.2, random_state=42)
train_positions = train_orig + [p + num_original_edges for p in train_orig]
test_positions = test_orig + [p + num_original_edges for p in test_orig]

model = RoadRiskGNN(input_dim=X.shape[1], hidden_dim=64)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=300, gamma=0.5)
loss_function = nn.MSELoss()

epochs = 1000
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    node_embeddings = model(data.x, data.edge_index)
    all_predictions = model.predict_edges(node_embeddings, data.edge_index)
    loss = loss_function(all_predictions[train_positions], y_edge[train_positions])
    loss.backward()
    optimizer.step()
    scheduler.step()
    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Training Loss: {loss.item():.5f}")


Epoch 0, Training Loss: 0.21836
Epoch 100, Training Loss: 0.00913
Epoch 200, Training Loss: 0.00913
Epoch 300, Training Loss: 0.00908


In [ ]:
# ===== EVALUATE =====
model.eval()
with torch.no_grad():
    node_embeddings = model(data.x, data.edge_index)
    all_predictions = model.predict_edges(node_embeddings, data.edge_index)
    test_predictions = all_predictions[test_positions]
    test_labels = y_edge[test_positions]
    test_loss = loss_function(test_predictions, test_labels)
    baseline_pred = y_edge[train_positions].mean()
    baseline_loss = loss_function(torch.full_like(test_labels, baseline_pred), test_labels)

print("Model test loss:   ", test_loss.item())
print("Baseline test loss:", baseline_loss.item())
print("Prediction spread:", test_predictions.std().item())
print("True label spread:", test_labels.std().item())

torch.save(model.state_dict(), "gnn_model.pth")

In [ ]:
# ===== EXPORT =====
original_edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
model.eval()
with torch.no_grad():
    node_embeddings = model(data.x, data.edge_index)
    final_predictions = model.predict_edges(node_embeddings, original_edge_index)

export_records = []
for i, (node_a, node_b) in enumerate(edges):
    export_records.append({
        "edge_id": i,
        "road_number": node_features_sorted.iloc[node_a]["road_number"],
        "start_lat": node_features_sorted.iloc[node_a]["latitude"],
        "start_lon": node_features_sorted.iloc[node_a]["longitude"],
        "end_lat": node_features_sorted.iloc[node_b]["latitude"],
        "end_lon": node_features_sorted.iloc[node_b]["longitude"],
        "predicted_risk": round(final_predictions[i].item(), 4)
    })

with open("gnn_risk_predictions.json", "w") as f:
    json.dump(export_records, f, indent=2)

print("Exported:", len(export_records), "segments")